# Week 12 · RealNVP on 2D blobs

**Mode: experiment · Sessions 6 and 7 · three short implementation exercises**

Use one notebook for both sessions. Keep this learner notebook as the exercise baseline; work in the paired solved notebook, which starts as an exact copy with the exercises open. Write your own observations after running it.

| Session | Route | Stop condition |
|---|---|---|
| 6 · 75–90 min | Make blobs → implement one coupling layer → stack layers → check inverse, Jacobian, and gradients | All checks above the Session 6 stop pass, including an automatic Jacobian check. No training required. |
| 7 · 60–90 min | Train six layers → inspect NLL and samples → train one layer as a controlled failure case | Finite train/held-out NLL, three plots inspected, and three short observations written. |

These are timeboxes, not deadlines. If debugging consumes a session, resume at the next cell later. Attractive samples and a perfect likelihood curve are not stop conditions.

We use the data-to-latent convention from the [Week 12 notes](../notes/w12_cs236_normalizing_flows.md) and [derivations](../proofs/w12_change_of_variables_flows.md):

$$
z=f_\theta(x),\qquad
\log p_\theta(x)=\log p_Z(f_\theta(x))+\log|\det J_{f_\theta}(x)|,\qquad
x=f_\theta^{-1}(z).
$$

The base distribution is $\mathcal N(0,I_2)$. Synthetic blobs keep the experiment local and reproducible. The notebook does not download data or add dependencies.


## Session 6 · 1. Make a small density problem

Four Gaussian blobs sit near the corners of a square. Each coordinate has a bimodal marginal, which makes the one-layer failure case easy to diagnose later. The labels below are used only to generate blobs; the flow trains without labels.


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn

SEED = 12
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(min(2, torch.get_num_threads()))

config = {
    "train_size": 2_000,
    "heldout_size": 400,
    "blob_std": 0.35,
    "layers": 6,
    "hidden": 32,
    "full_steps": 450,
    "one_layer_steps": 200,
    "batch_size": 256,
    "learning_rate": 2e-3,
}
centers = np.array(
    [[-1.8, -1.8], [-1.8, 1.8], [1.8, -1.8], [1.8, 1.8]],
    dtype=np.float32,
)
n = config["train_size"] + config["heldout_size"]
labels = rng.integers(0, len(centers), size=n)
points = centers[labels] + rng.normal(0, config["blob_std"], size=(n, 2)).astype(np.float32)
order = rng.permutation(n)
train_x = torch.from_numpy(points[order[:config["train_size"]]].copy())
heldout_x = torch.from_numpy(points[order[config["train_size"]:]].copy())
assert train_x.shape == (2_000, 2) and heldout_x.shape == (400, 2)
assert torch.isfinite(train_x).all()
print(f"{len(train_x)} train and {len(heldout_x)} held-out points; CPU training.")


In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(train_x[:, 0], train_x[:, 1], s=4, alpha=0.35)
ax.set(title="Training data: four 2D blobs", xlabel="$x_1$", ylabel="$x_2$", xlim=(-3.5, 3.5), ylim=(-3.5, 3.5))
ax.set_aspect("equal")
plt.show()


## Session 6 · 2. Implement one affine coupling layer

For a chosen transformed coordinate $j$, let $c$ be the other coordinate. The scale and shift network reads only $x_c$:

$$
y_c=x_c,\qquad
y_j=x_j e^{s(x_c)}+t(x_c),\qquad
\log|\det J_{x\to y}|=s(x_c).
$$

The supplied network bounds $s$ to $[-0.8,0.8]$ for numerical stability. Its final layer starts at zero, so a fresh coupling layer begins as the identity.

**TODO 1 of 3:** In forward, compute $y$ and return the per-point log-determinant with shape (batch,). Do not use a full Jacobian in the implementation.

**TODO 2 of 3:** In inverse, undo the shift and scale. The conditioning coordinate stays unchanged.


In [ ]:
class AffineCoupling(nn.Module):
    """Transform one of two coordinates, conditioned on the other."""

    def __init__(self, transformed: int, hidden: int = 32):
        super().__init__()
        assert transformed in (0, 1)
        self.transformed = transformed
        self.conditioning = 1 - transformed
        self.net = nn.Sequential(
            nn.Linear(1, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, 2),
        )
        nn.init.zeros_(self.net[-1].weight)
        nn.init.zeros_(self.net[-1].bias)

    def scale_shift(self, condition: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        raw_scale, shift = self.net(condition).chunk(2, dim=1)
        return 0.8 * torch.tanh(raw_scale), shift

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        assert x.ndim == 2 and x.shape[1] == 2
        # TODO 1: preserve the conditioning coordinate, transform the other,
        # and return (y, log_det) with shapes (batch, 2) and (batch,).
        raise NotImplementedError("TODO 1: implement coupling forward and log-determinant.")

    def inverse(self, y: torch.Tensor) -> torch.Tensor:
        assert y.ndim == 2 and y.shape[1] == 2
        # TODO 2: recover x using (y_j - t(y_c)) * exp(-s(y_c)).
        raise NotImplementedError("TODO 2: implement coupling inverse.")


### Check one layer before stacking

Set a nonzero scale and shift so the test cannot pass merely because the layer starts as the identity. Compare the reported log-determinant with an automatic Jacobian of one point.


In [ ]:
probe_layer = AffineCoupling(transformed=1)
with torch.no_grad():
    probe_layer.net[-1].bias.copy_(torch.tensor([0.6, 0.25]))
probe = torch.tensor([[0.4, -1.2], [1.0, 2.0]])
mapped, log_det = probe_layer(probe)
scale, shift = probe_layer.scale_shift(probe[:, :1])
assert mapped.shape == probe.shape and log_det.shape == (len(probe),)
assert torch.allclose(mapped[:, :1], probe[:, :1])
assert torch.allclose(mapped[:, 1:2], probe[:, 1:2] * scale.exp() + shift)
assert torch.allclose(log_det, scale.squeeze(1))
assert torch.allclose(probe_layer.inverse(mapped), probe, atol=1e-6)
jac = torch.autograd.functional.jacobian(
    lambda point: probe_layer(point.unsqueeze(0))[0].squeeze(0), probe[0]
)
assert torch.allclose(torch.linalg.slogdet(jac)[1], log_det[0], atol=1e-6)
print("One layer: forward, inverse, shape, and analytic log-determinant checks passed.")


## Session 6 · 3. Compose alternating layers

The first layer changes coordinate 2, the next changes coordinate 1, and so on. With column-vector Jacobians, the Jacobian of layer 2 after layer 1 is $J_2(y)J_1(x)$. Their **log absolute determinants add**, evaluated at each layer's input.

The flow's forward direction maps data to the Gaussian base. Inverse applies layer inverses in reverse order. A negative log probability is the training loss.


In [ ]:
class RealNVP2D(nn.Module):
    """An alternating stack mapping data x to Gaussian latent z."""

    def __init__(self, layers: int = 6, hidden: int = 32):
        super().__init__()
        self.layers = nn.ModuleList(
            AffineCoupling(transformed=(i + 1) % 2, hidden=hidden)
            for i in range(layers)
        )
        self.base = torch.distributions.Normal(0.0, 1.0)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        z = x
        total_log_det = x.new_zeros(len(x))
        for layer in self.layers:
            z, log_det = layer(z)
            total_log_det = total_log_det + log_det
        return z, total_log_det

    def inverse(self, z: torch.Tensor) -> torch.Tensor:
        x = z
        for layer in reversed(self.layers):
            x = layer.inverse(x)
        return x

    def log_prob(self, x: torch.Tensor) -> torch.Tensor:
        # TODO 3: evaluate the standard-normal base density of z,
        # sum its two coordinate log densities, and add the forward log-det.
        raise NotImplementedError("TODO 3: implement log_prob.")

    @torch.no_grad()
    def sample(self, n: int, generator: torch.Generator | None = None) -> torch.Tensor:
        z = torch.randn(n, 2, generator=generator)
        return self.inverse(z)


### Check the stack before training

This disposable model uses small nonzero last-layer weights. It checks round trips, the density sign, a full automatic Jacobian, and gradients. Stop and inspect the relevant TODO if a check fails.


In [ ]:
torch.manual_seed(SEED + 1)
smoke = RealNVP2D(layers=4)
with torch.no_grad():
    for layer in smoke.layers:
        layer.net[-1].weight.normal_(0, 0.03)
        layer.net[-1].bias.copy_(torch.tensor([0.1, -0.05]))
test_points = torch.tensor([[0.4, -1.2], [1.0, 2.0]])
test_z, test_log_det = smoke(test_points)
assert test_z.shape == test_points.shape and test_log_det.shape == (2,)
assert torch.allclose(smoke.inverse(test_z), test_points, atol=1e-5)
expected = smoke.base.log_prob(test_z).sum(dim=1) + test_log_det
assert torch.allclose(smoke.log_prob(test_points), expected)
full_jac = torch.autograd.functional.jacobian(
    lambda point: smoke(point.unsqueeze(0))[0].squeeze(0), test_points[0]
)
assert torch.allclose(torch.linalg.slogdet(full_jac)[1], test_log_det[0], atol=1e-5)
loss = -smoke.log_prob(test_points).mean()
loss.backward()
gradients = [p.grad for p in smoke.parameters() if p.grad is not None]
assert gradients and all(torch.isfinite(g).all() for g in gradients)
assert sum(g.abs().sum().item() for g in gradients) > 0
print("Stack: round trip, density, total Jacobian, and gradient checks passed.")


## Session 6 stop · The flow is mathematically wired

Stop here when all checks pass. You have a data-to-latent forward map, a latent-to-data inverse, a log probability, and nonzero gradients. Training and plot interpretation belong to Session 7. If you restart the kernel next session, run the cells above from the top.


## Session 7 · 4. Train the six-layer flow

The same synthetic train/held-out split is used throughout. Each run starts a fresh model and seeded minibatch draws. Held-out NLL is evaluated without gradient updates every 25 steps. We do not tune hyperparameters against the held-out set in this exercise.


In [ ]:
def fit_flow(layers: int, steps: int, seed: int) -> tuple[RealNVP2D, np.ndarray]:
    """Train a fresh flow and record train/held-out NLL."""
    torch.manual_seed(seed)
    model = RealNVP2D(layers=layers, hidden=config["hidden"])
    optimizer = torch.optim.Adam(model.parameters(), lr=config["learning_rate"])
    batch_rng = torch.Generator().manual_seed(seed + 1)
    history = []
    for step in range(1, steps + 1):
        model.train()
        indices = torch.randint(len(train_x), (config["batch_size"],), generator=batch_rng)
        loss = -model.log_prob(train_x[indices]).mean()
        assert torch.isfinite(loss), "Non-finite loss: check log_prob and scales."
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if step == 1 or step % 25 == 0 or step == steps:
            model.eval()
            with torch.no_grad():
                heldout_nll = -model.log_prob(heldout_x).mean().item()
            history.append((step, loss.item(), heldout_nll))
    return model.eval(), np.asarray(history)

flow, history = fit_flow(config["layers"], config["full_steps"], SEED + 10)
print(f"Six layers: held-out NLL {history[0, 2]:.2f} → {history[-1, 2]:.2f}")
assert np.isfinite(history).all()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(history[:, 0], history[:, 1], label="minibatch train NLL", alpha=0.7)
ax.plot(history[:, 0], history[:, 2], label="held-out NLL", linewidth=2)
ax.set(xlabel="Optimizer step", ylabel="Negative log likelihood", title="Six-layer RealNVP")
ax.legend()
plt.show()


## Session 7 · 5. Inspect both directions

The left panel shows held-out data. The middle panel maps held-out data through forward into latent space; it need not look perfectly Gaussian after a short run. The right panel draws new points by sampling the Gaussian base and applying inverse. Samples are unconditional: no input data point is supplied.


In [ ]:
with torch.no_grad():
    latent, _ = flow.forward(heldout_x)
    generated = flow.sample(800, generator=torch.Generator().manual_seed(SEED + 4))
assert latent.shape == heldout_x.shape and generated.shape == (800, 2)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
for ax, points, title in zip(
    axes,
    [heldout_x, latent, generated],
    ["Held-out data", "Data mapped to latent", "Fresh generated samples"],
):
    values = points.detach().numpy()
    ax.scatter(values[:, 0], values[:, 1], s=7, alpha=0.45)
    ax.set(title=title, xlabel="Coordinate 1", ylabel="Coordinate 2")
    ax.set_aspect("equal")
plt.tight_layout()
plt.show()


## Session 7 · 6. One controlled failure: only one coupling layer

One data-to-latent coupling layer transforms coordinate 2 and leaves coordinate 1 unchanged. Its inverse therefore leaves the first Gaussian base coordinate unchanged too. Train it on the **same** data and compare both sample clouds and the first-coordinate marginal. This failure follows from the architecture even if its optimizer runs perfectly.


In [ ]:
one_layer, one_history = fit_flow(1, config["one_layer_steps"], SEED + 20)
with torch.no_grad():
    base_draws = torch.randn(1_000, 2, generator=torch.Generator().manual_seed(SEED + 5))
    one_samples = one_layer.inverse(base_draws)
    full_samples = flow.inverse(base_draws)
    full_nll = -flow.log_prob(heldout_x).mean().item()
    one_nll = -one_layer.log_prob(heldout_x).mean().item()
assert torch.allclose(one_samples[:, 0], base_draws[:, 0], atol=1e-6)
assert np.isfinite([full_nll, one_nll]).all()
print(f"Held-out NLL: six layers {full_nll:.2f}; one layer {one_nll:.2f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(heldout_x[:, 0], heldout_x[:, 1], s=5, alpha=0.25, label="data")
axes[0].scatter(one_samples[:, 0], one_samples[:, 1], s=5, alpha=0.25, label="one-layer samples")
axes[0].set(title="One layer: first coordinate stays Gaussian", xlabel="$x_1$", ylabel="$x_2$")
axes[0].legend(markerscale=2)
bins = np.linspace(-3.5, 3.5, 50)
axes[1].hist(heldout_x[:, 0].numpy(), bins=bins, density=True, alpha=0.5, label="data")
axes[1].hist(one_samples[:, 0].numpy(), bins=bins, density=True, alpha=0.5, label="one layer")
axes[1].hist(full_samples[:, 0].numpy(), bins=bins, density=True, histtype="step", linewidth=2, label="six layers")
axes[1].set(title="First-coordinate marginal", xlabel="$x_1$", ylabel="Density")
axes[1].legend()
plt.tight_layout()
plt.show()


## Session 7 stop · Record three observations

Write one or two sentences for each prompt below, using the actual output you see:

1. **Likelihood:** How did held-out NLL change? Did the train and held-out curves tell the same story?
2. **Generated distribution:** Compare generated samples with the target blobs. Name one visible success and one gap; do not infer density quality from samples alone.
3. **Failure mechanism:** What does the one-layer model fail to change about coordinate 1? How do the marginal plot and NLL support your account?

**Done means:** all checks pass, the short runs finish with finite losses, you inspect the curves and two sample comparisons, and you write these observations. The one-layer failure is a controlled architecture comparison, not evidence that every trained six-layer flow perfectly models the data.


### Your observations

1. **Likelihood:** …
2. **Generated distribution:** …
3. **One-layer failure:** …
